# Phase 2
## Data Preprocessing and Cleaning in Fraud Detection ML

In [ ]:
# Removing duplicates
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv(r"C:\Users\u\Downloads\creditcard.csv\creditcard.csv")

# Remove duplicate
before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Removed:     {before - after:,} duplicate rows")

In [ ]:
# Separate features (X) from target (y)
# X = everything EXCEPT the Class column — these are the inputs
# y = only the Class column — this is what we want to predict
X = df.drop('Class', axis=1)    # axis=1 means "drop this column"
y = df['Class']                 # Series of 0s and 1s

print(f"X shape: {X.shape}")    #(283726, 30)  - 30 features
print(f"y shape: {y.shape}")    #(283726)    - 1 target

print(f"\nClass distribution in y:")
print(y.value_counts())
print(y.value_counts(normalize=True).mul(100).round(3))


In [ ]:
# Train/test split before scaling --- this order matters
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% goes to test, 80% stays for training
    random_state=42,    # "Seed" - makes the split reproducible every run
    stratify=y          # Critical for imbalanced data
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# Verify the fraud ratio is preserved in both splits
print(f"\nFraud % in training set: {y_train.mean()*100:.4f}%")
print(f"Fraud % in test set:    {y_test.mean()*100:.4f}%")

In [ ]:
# Feature scaling with StandardScaler
from sklearn.preprocessing import StandardScaler

# Create the scaler -- we only scale Amount and Time
scaler = StandardScaler()

X_train[['scaled_amount', 'scaled_time']] = scaler.fit_transform(
    X_train[['Amount', 'Time']]  # fit + transform in one step on Training data
)
X_test[['scaled_amount', 'scaled_time']] = scaler.transform(
    X_test[['Amount', 'Time']]    # transform only -- no fitting on test data
)

# Drop tne original raw columns
#X_train = X_train.drop(['Amount', 'Time'], axis=1)
#X_test = X_test.drop(['Amount', 'Time'], axis=1)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# Verify scaling worked on training data
print("\nAfter scaling - training set statistics:")
print(X_train[['scaled_amount', 'scaled_time']].describe().round(4))

In [ ]:
# Verify everything looks correct
# Checking preprocessing output before moving on
# Sanity check 1: no missing values introduced

print("Missing values after preprocessing:")
print(f" X_train: {X_train.isnull().sum().sum()}")
print(f" X_test:  {X_test.isnull().sum().sum()}")
print(f" y_train: {y_train.isnull().sum()}")
print(f" y_test:  {y_test.isnull().sum()}")

# Sanity check 2 -- Class ratios preserved
print("\nClass ratio check:")
print(f" y_train fraud rate: {y_train.mean()*100:.4f}%")
print(f" y_test  fraud rate: {y_test.mean()*100:.4f}%")

# Sanity check 3
print("\nColumn names in X_train:")
print(X_train.columns.tolist())

# 'Amount' and 'Time' should NOT appear — only 'scaled_amount', 'scaled_time'

# Sanity check 4
print("\nScaled feature statistics:")
print(X_train.describe().round(3))

# Sanity check 5: shapes are consistent
print("\nShape check:")
assert X_train.shape[0] == y_train.shape[0],   "Train rows dont match!"
assert X_test.shape[0]  == y_test.shape[0],    "Test rows dont match!"
assert X_train.shape[1] == X_test.shape[1],    "Column count mismatch"
print("All shape checks passed.")

In [ ]:
# Save the preprocessed data
# ─── SAVE PREPROCESSED SPLITS TO DISK ─────────────────────────────────────
# This lets you resume from Phase 3 without re-running Phase 1 and 2

import os
os.makedirs('preprocessed', exist_ok=True)

X_train.to_csv('preprocessed/X_train.csv', index=False)
X_test.to_csv('preprocessed/X_test.csv',   index=False)
y_train.to_csv('preprocessed/y_train.csv', index=False)
y_test.to_csv('preprocessed/y_test.csv',   index=False)

# Save the scaler too - needed when deploying
import joblib
joblib.dump(scaler, 'preprocessed/scaler.pkl')

print("Preprocessed data saved to /preprocessed/")
print("Scaler saved to preprocessed/scaler.pkl")


# Later, to reload everything:
# X_train = pd.read_csv('preprocessed/X_train.csv')
# scaler  = joblib.load('preprocessed/scaler.pkl')